In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

In [14]:
nb_labeled = pd.read_csv("../../data/clean/nb_labeled.csv")
nb_labeled.head().T

,0,1,2,3,4
participant_id,P1,P1,P1,P1,P1
calendarDate,2025-07-13,2025-08-18,2025-08-19,2025-08-20,2025-08-21
cohort,cohort_03,cohort_03,cohort_03,cohort_03,cohort_03
durationInMs,28620000.0,28500000.0,26520000.0,25500000.0,23460000.0
deepSleepDurationInMs,3480000.0,5160000.0,2160000.0,4620000.0,2820000.0
lightSleepDurationInMs,20160000.0,20100000.0,20040000.0,16980000.0,19620000.0
remSleepInMs,4980000.0,3240000.0,4320000.0,3900000.0,1020000.0
awakeDurationInMs,360000.0,780000.0,180000.0,0.0,900000.0
overallSleepScore,79.0,79.0,79.0,80.0,48.0
breathsPerMinute_mean,19.0785,19.163504,18.843708,17.915741,18.371798


In [15]:
y = nb_labeled["low_recovery"]
groups = nb_labeled["participant_id"]

features_to_remove = [
    "participant_id",
    "cohort",
    "calendarDate",
    "low_recovery"
]

X = nb_labeled.drop(columns = features_to_remove)

print(y.shape,X.shape)


(18728,) (18728, 20)


In [ ]:
# Train/test split pattern adapted from the scikit-learn StratifiedGroupKFold documentation example
# modified to use a single split and this project's columns.

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(sgkf.split(X,y,groups = groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train size: {len(train_idx)}")
print(f"Test size: {len(test_idx)}")

Train size: 15003
Test size: 3725


In [23]:
print(f"Training: {groups.iloc[train_idx].nunique()} people\n Testing: {groups.iloc[test_idx].nunique()} people")

Training: 79 people
 Testing: 19 people


In [22]:
print(y_test.mean()*100, y_train.mean()*100)

10.228187919463087 10.257948410317937


In [31]:
print(f"Training: {groups.iloc[train_idx].nunique()} people, {len(train_idx)} nights, {(y_train.mean()*100).round(2)} low recovery")
print(f"Testing: {groups.iloc[test_idx].nunique()} people, {len(test_idx)} nights, {(y_test.mean()*100).round(2)} low recovery")
print("With training and test split such that there are 79 + 19 = 98 (100%) people and 150003 + 3725 = 18728 (100%) nights, " \
"\n we can conclude that no person appears in both sets")

Training: 79 people, 15003 nights, 10.26 low recovery
Testing: 19 people, 3725 nights, 10.23 low recovery
With training and test split such that there are 79 + 19 = 98 (100%) people and 150003 + 3725 = 18728 (100%) nights, 
 we can conclude that no person appears in both sets


Training: 79 people, 15,003 nights, 10.23% low recovery.
Testing: 19 people, 3,725 nights, 10.26% low recovery.
No person appears in both sets.

In [32]:
nb_labeled.iloc[train_idx].to_csv("../../data/clean/nb_train.csv")
nb_labeled.iloc[test_idx].to_csv("../../data/clean/nb_test.csv")